# Week 39, Day 5: Efficient coding of natural sounds

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w039/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

This lesson uses a dataset from the Engram site. Run this cell first: it downloads the file once and defines `dataset()`.

In [ ]:
import os, urllib.request
def dataset(name):
    """Downloads an Engram dataset the first time it is used and returns its local path."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://engram-neuroai.vercel.app/data/" + name, name)
    return name

### Exercise: Principal component whitening

Write whiten(X, k): with X of shape (64, N) and mean already removed, compute the covariance, its eigenvectors E and eigenvalues ev, keep the k largest, and return Z = V @ X with V = diag(1 / sqrt(ev)) @ E.T, together with V.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2002)
fs, L, N = 16000, 64, 6000
t = np.arange(L) / fs

def tonal():
    x = np.zeros(L)
    for _ in range(3):
        f = r.uniform(300, 6000); x += r.normal() * np.sin(2 * np.pi * f * t + r.uniform(0, 2 * np.pi))
    return x

def transient():
    x = np.zeros(L)
    for _ in range(2):
        t0 = r.integers(0, L); f = r.uniform(300, 6000); tau = r.uniform(0.0002, 0.0008); tt = (np.arange(L) - t0) / fs
        x += r.normal() * np.where(tt >= 0, np.exp(-np.maximum(tt, 0) / tau) * np.sin(2 * np.pi * f * tt), 0)
    return x

def dataset(gen):
    X = np.array([gen() for _ in range(N)]).T
    return X - X.mean(1, keepdims=True)

X_tone, X_trans = dataset(tonal), dataset(transient)

def whiten(X, k):
    C = X @ X.T / X.shape[1]
    ev, E = np.linalg.eigh(C)
    idx = np.argsort(ev)[::-1][:k]
    V = ...
    return V @ X, V

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for a, X, name in zip(ax, (X_tone, X_trans), ("tonal", "transient")):
    for j in range(3): a.plot(t * 1000, X[:, j] + 3 * j)
    a.set_title(name); a.set_xlabel("ms")
plt.show()

In [ ]:
import numpy as np
def test_identity():
    "the whitened data have covariance close to the identity"
    Z, V = whiten(X_tone, 32)
    C = Z @ Z.T / Z.shape[1]
    assert np.allclose(C, np.eye(32), atol=1e-6)

test_identity()
print("All checks passed")

### Exercise: FastICA from scratch

Complete the FastICA update inside the loop: compute Y = W @ Z, gY = tanh(Y), the new W from the formula, then symmetric decorrelation with the SVD (U @ Vt from np.linalg.svd(W_new)).

In [ ]:
from scipy.stats import kurtosis

def fastica(Z, k, iters=200, seed=0):
    g = np.random.default_rng(seed)
    W = np.linalg.qr(g.normal(size=(k, k)))[0]
    for it in range(iters):
        ...
        if np.max(np.abs(np.abs(np.diag(W_new @ W.T)) - 1)) < 1e-6:
            return W_new, it
        W = W_new
    return W, iters

Z_t, V_t = whiten(X_trans, 32)
W_t, n_it = fastica(Z_t, 32)
kurt_ica = np.mean(kurtosis(W_t @ Z_t, axis=1))
kurt_pca = np.mean(kurtosis(Z_t, axis=1))
print(f"converged after {n_it} iterations; mean excess kurtosis: PCA components {kurt_pca:.2f}, ICA components {kurt_ica:.2f}")

In [ ]:
import numpy as np
def test_orthonormal():
    "the learned W is orthonormal"
    W, _ = fastica(Z_t, 32, iters=20)
    assert np.allclose(W @ W.T, np.eye(32), atol=1e-6)

def test_sparser():
    "ICA components are heavier-tailed than principal components (mean excess kurtosis at least 1 higher)"
    assert kurt_ica > kurt_pca + 1

test_orthonormal()
test_sparser()
print("All checks passed")

### Exercise: Fourier-like and wavelet-like

Write basis(X): whiten to 32 dimensions, run fastica, and return the basis functions as the columns of the pseudo-inverse of W @ V. Write spread(a) as described. Then compute the median spread for tonal, transient and an equal-power 50:50 mixed dataset, and the fraction of basis functions shorter than 10 samples.

In [ ]:
def basis(X, k=32):
    ...

def spread(a):
    p = a ** 2 / np.sum(a ** 2)
    m = np.sum(p * np.arange(L))
    return ...

X_mix = np.concatenate([X_tone[:, : N // 2] / X_tone.std(), X_trans[:, : N // 2] / X_trans.std()], axis=1)   # equal power
spread_med, frac_short, bases = {}, {}, {}
for name, X in (("tonal", X_tone), ("transient", X_trans), ("mixed", X_mix)):
    bases[name] = basis(X)
    sp = np.array([spread(bases[name][:, j]) for j in range(bases[name].shape[1])])
    spread_med[name], frac_short[name] = np.median(sp), np.mean(sp < 10)
print({k: round(float(v), 1) for k, v in spread_med.items()}, "samples (median temporal spread)")
print({k: f"{v:.0%}" for k, v in frac_short.items()}, "of basis functions shorter than 10 samples")
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for a, name in zip(ax, ("tonal", "transient")):
    for j in range(4): a.plot(t * 1000, bases[name][:, j] / np.abs(bases[name][:, j]).max() + 2.2 * j)
    a.set_title(name); a.set_xlabel("ms")
plt.show()

In [ ]:
import numpy as np
def test_spread():
    "tonal basis functions are at least twice as spread out in time as transient ones"
    assert spread_med["tonal"] > 2 * spread_med["transient"]

def test_mixture():
    "an equal-power mixture gives some short and some long basis functions, and an intermediate median spread"
    assert 0 < frac_short["mixed"] < 1 and spread_med["transient"] < spread_med["mixed"] < spread_med["tonal"] - 2

test_spread()
test_mixture()
print("All checks passed")